# Step1: Setup and Configuration

In [ ]:
#Uncomment the following lines to install required packages if you are running this notebook in a new environment.
# Install required packages (this may take 2-3 minutes)
# Installing in compatible groups to avoid dependency conflicts
# %pip install --upgrade --quiet "google-cloud-aiplatform>=1.38.0" "kfp>=2.4.0"
# %pip install --upgrade --quiet "scikit-learn>=1.4.2" "imbalanced-learn>=0.12.0"
# %pip install --upgrade --quiet pandas numpy

# print("\n" + "="*70)
# print(" Installation complete!")
# print("="*70)
# print("\n NEXT STEPS:")
# print("1. Restart the kernel: Kernel → Restart Kernel")
# print("2. Skip this cell (don't run it again)")
# print("3. Continue from cell 2 (Import libraries)")
# print("="*70)

In [10]:
import google.auth
import google.cloud.aiplatform
import kfp
import sklearn
import imblearn
import pandas
import numpy

print("google-auth:", google.auth.__version__)
print("aiplatform:", google.cloud.aiplatform.__version__)
print("kfp:", kfp.__version__)
print("sklearn:", sklearn.__version__)
print("imblearn:", imblearn.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)

print(" Imports successful")

google-auth: 2.59.1
aiplatform: 2.3.0
kfp: 2.17.0
sklearn: 1.9.1
imblearn: 0.14.2
pandas: 3.0.6
numpy: 2.5.3
 Imports successful


# Importing Libraries and Initializing vertex AI

In [11]:
# Import libraries
import os
import json
from datetime import datetime
from typing import NamedTuple

from google.cloud import aiplatform
from kfp import dsl
from kfp.dsl import Output, Input, Dataset, Model, Metrics, component
from kfp import compiler
import pandas as pd
import numpy as np

# Configuration - UPDATE THESE VALUES
PROJECT_ID = "mlopstest-510202"
REGION = "northamerica-northeast1"           # Choose your preferred region
BUCKET_NAME = f"{PROJECT_ID}-vertex-pipelines"  # GCS bucket for artifacts
PIPELINE_ROOT = f"gs://{BUCKET_NAME}/pipeline_root"

# Initialize Vertex AI
aiplatform.init(project=PROJECT_ID, location=REGION)

print(f" Project ID: {PROJECT_ID}")
print(f" Region: {REGION}")
print(f" Pipeline Root: {PIPELINE_ROOT}")

 Project ID: mlopstest-510202
 Region: northamerica-northeast1
 Pipeline Root: gs://mlopstest-510202-vertex-pipelines/pipeline_root


# Create GCS bucket if it doesn't exist

In [12]:
!gsutil ls -b gs://{BUCKET_NAME} || gsutil mb -l {REGION} gs://{BUCKET_NAME}
print(f"Bucket ready: gs://{BUCKET_NAME}")

gs://mlopstest-510202-vertex-pipelines/
Bucket ready: gs://mlopstest-510202-vertex-pipelines


# Grant service account permissions to access the bucket
# Get the project number for the service account

In [16]:
!gcloud services enable cloudresourcemanager.googleapis.com --project={PROJECT_ID}

result = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
PROJECT_NUMBER = result[0] if result else None

if PROJECT_NUMBER:
    SERVICE_ACCOUNT = f"{PROJECT_NUMBER}-compute@developer.gserviceaccount.com"
    print(f"Granting permissions to service account: {SERVICE_ACCOUNT}")

    !gsutil iam ch serviceAccount:{SERVICE_ACCOUNT}:roles/storage.objectAdmin gs://{BUCKET_NAME}

    print("Permissions granted successfully!")
    print(f"Service account can now read/write to gs://{BUCKET_NAME}")
else:
    print("Could not determine project number.")

Granting permissions to service account: 288053996263-compute@developer.gserviceaccount.com
Permissions granted successfully!
Service account can now read/write to gs://mlopstest-510202-vertex-pipelines


No changes made to gs://mlopstest-510202-vertex-pipelines/
